In [1]:
import sys
print(sys.version)
print(sys.platform)

3.13.9 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 19:09:58) [MSC v.1929 64 bit (AMD64)]
win32


In [2]:
import asyncio
import sys

asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

loop = asyncio.new_event_loop()
asyncio.set_event_loop(loop)

print("Policy:", type(asyncio.get_event_loop_policy()).__name__)
print("Loop:", type(asyncio.get_event_loop()).__name__)

Policy: WindowsProactorEventLoopPolicy
Loop: _WindowsSelectorEventLoop


In [3]:
import nest_asyncio

# Force-replace the running loop with a fresh Proactor loop
loop = asyncio.ProactorEventLoop()
asyncio.set_event_loop(loop)
nest_asyncio.apply(loop)

print("Loop:", type(asyncio.get_event_loop()).__name__)

Loop: _WindowsSelectorEventLoop


In [4]:
import asyncio

# Forcefully close and replace the existing loop
old_loop = asyncio.get_event_loop()
old_loop.close()

new_loop = asyncio.ProactorEventLoop()
asyncio.set_event_loop(new_loop)

import nest_asyncio
nest_asyncio.apply(new_loop)

print("Loop:", type(asyncio.get_event_loop()).__name__)

RuntimeError: Cannot close a running event loop

In [5]:
from playwright.sync_api import sync_playwright

def test():
    with sync_playwright() as p:
        browser = p.chromium.launch(headless=True)
        page = browser.new_page()
        page.goto("https://example.com")
        print("Title:", page.title())
        browser.close()

import threading
t = threading.Thread(target=test)
t.start()
t.join()

Title: Example Domain


In [9]:
import re
import time
import threading
import pandas as pd
from playwright.sync_api import sync_playwright

MIN_YEAR   = 2018
MAX_PAGES  = 3
OUTPUT_CSV = "used_car_export_data.csv"
DATA       = []

def clean_number(text):
    if not text:
        return None
    m = re.findall(r"[\d,]+", text)
    return m[0].replace(",", "") if m else None

def extract_year(text):
    m = re.search(r"\b(20\d{2})\b", text or "")
    return int(m.group(1)) if m else None

def safe_text(card, selector):
    try:
        el = card.query_selector(selector)
        return el.inner_text().strip() if el else None
    except:
        return None

def parse_card(card, fields, source):
    try:
        full_text = card.inner_text()
        year = extract_year(full_text)
        if year and year < MIN_YEAR:
            return None
        return {
            "make_model":   safe_text(card, fields["make_model"]) or full_text.split("\n")[0].strip(),
            "year":         year,
            "price":        clean_number(safe_text(card, fields["price"])),
            "mileage_km":   clean_number(safe_text(card, fields["mileage"])),
            "engine_cc":    clean_number(safe_text(card, fields["engine_size"])),
            "fuel_type":    safe_text(card, fields["fuel_type"]),
            "transmission": safe_text(card, fields["transmission"]),
            "body_type":    safe_text(card, fields["body_type"]),
            "source":       source,
        }
    except:
        return None

print("Functions defined")

Functions defined


In [10]:
PLATFORMS = [
    {
        "name": "SBT Japan",
        "url":  "https://www.sbtjapan.com/used-cars-for-sale",
        "card_selector": "li.car-list-item, div.vehicle-card, div.stockList_item",
        "next_selector": "a.next, a[rel='next'], li.next > a",
        "fields": {
            "make_model":   "h2.car-name, .vehicle-title, .car-title",
            "price":        ".price, .car-price, span[class*='price']",
            "mileage":      ".mileage, span[class*='mileage']",
            "engine_size":  ".engine, span[class*='engine']",
            "fuel_type":    ".fuel, span[class*='fuel']",
            "transmission": ".transmission, span[class*='transmission']",
            "body_type":    ".body-type, span[class*='body']",
        },
    },
    {
        "name": "AutoCJ",
        "url":  "https://autocj.co.jp/stocklist/",
        "card_selector": "div.car-box, div.stock-item, article.car-card",
        "next_selector": "a.next, .pagination a[rel='next']",
        "fields": {
            "make_model":   ".car-name, h3.title",
            "price":        ".price",
            "mileage":      ".mileage",
            "engine_size":  ".engine",
            "fuel_type":    ".fuel",
            "transmission": ".transmission",
            "body_type":    ".body",
        },
    },
    {
        "name": "AAA Japan",
        "url":  "https://www.aaajapan.com/stock/",
        "card_selector": "div.car-item, div.stock-car, div.vehicle-list-item",
        "next_selector": "a.next, .pagination .next",
        "fields": {
            "make_model":   ".car-name, h2",
            "price":        ".price",
            "mileage":      ".mileage",
            "engine_size":  ".engine",
            "fuel_type":    ".fuel",
            "transmission": ".transmission",
            "body_type":    ".body-type",
        },
    },
]

print(f" {len(PLATFORMS)} platforms configured")

 3 platforms configured


In [11]:
def scrape_site(platform, max_pages=MAX_PAGES):
    name    = platform["name"]
    results = []

    with sync_playwright() as p:
        browser = p.chromium.launch(headless=True)
        context = browser.new_context(user_agent=(
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/124.0.0.0 Safari/537.36"
        ))
        page = context.new_page()

        try:
            page.goto(platform["url"], timeout=60000, wait_until="domcontentloaded")
        except Exception as e:
            print(f"  ❌ {name}: failed to load — {e}")
            browser.close()
            return results

        for page_num in range(1, max_pages + 1):
            print(f"  📄 {name} — page {page_num}")

            try:
                page.wait_for_selector(platform["card_selector"], timeout=10000)
            except:
                print(f"     No cards found — stopping")
                break

            cards = page.query_selector_all(platform["card_selector"])
            print(f"     Found {len(cards)} cards")

            for card in cards:
                car = parse_card(card, platform["fields"], name)
                if car:
                    results.append(car)

            try:
                next_btn = page.query_selector(platform["next_selector"])
                if not next_btn or not next_btn.is_visible():
                    print(f"     No next button — done")
                    break
                next_btn.click()
                page.wait_for_load_state("domcontentloaded")
                page.wait_for_timeout(2000)
            except Exception as e:
                print(f"     Pagination error: {e}")
                break

        browser.close()
    return results

print(" scrape_site defined")


 scrape_site defined


In [12]:
def run_all():
    for platform in PLATFORMS:
        print(f"\n🚗 Scraping {platform['name']} ...")
        try:
            t_results = []
            def task():
                t_results.extend(scrape_site(platform))
            t = threading.Thread(target=task)
            t.start()
            t.join()
            DATA.extend(t_results)
            print(f"   ✅ {len(t_results)} cars collected")
        except Exception as e:
            print(f"   ❌ {platform['name']} failed: {e}")

    print(f"\n📦 Total cars collected: {len(DATA)}")

run_all()


🚗 Scraping SBT Japan ...
  📄 SBT Japan — page 1
     No cards found — stopping
   ✅ 0 cars collected

🚗 Scraping AutoCJ ...
  📄 AutoCJ — page 1
     No cards found — stopping
   ✅ 0 cars collected

🚗 Scraping AAA Japan ...
  📄 AAA Japan — page 1
     No cards found — stopping
   ✅ 0 cars collected

📦 Total cars collected: 0


In [14]:
import subprocess, sys

script = """
import re
from playwright.sync_api import sync_playwright

with sync_playwright() as p:
    browser = p.chromium.launch(headless=True)
    page = browser.new_page(user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36")
    page.goto("https://www.sbtjapan.com/used-cars-for-sale", timeout=60000, wait_until="domcontentloaded")
    page.wait_for_timeout(4000)
    html = page.content()
    print("TITLE:", page.title())
    print("HTML_LEN:", len(html))
    keywords = ["car", "vehicle", "stock", "list", "item", "card", "product"]
    found = set()
    for kw in keywords:
        for m in re.findall(rf'class="([^"]*{kw}[^"]*)"', html, re.IGNORECASE):
            for cls in m.split():
                found.add(cls)
    print("CLASSES:")
    for c in sorted(found)[:30]:
        print(" ", c)
    print("SNIPPET:")
    print(html[2000:4000])
    browser.close()
"""

result = subprocess.run(
    [sys.executable, "-c", script],
    capture_output=True, text=True, timeout=120
)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr[-2000:])

TITLE: Page does not exist | Global Car Exporters - SBT Japan
HTML_LEN: 449630
CLASSES:
  -brands
  -filter
  -four-items
  -parters-stock
  -sbt-stock
  -select
  -three-items
  -types
  car-choose
  car-choose-heading
  car-choose-inner-container
  car-choose__category
  car-choose__category-head
  car-choose__category-image
  car-choose__category-info
  car-choose__category-inner
  car-choose__category-item
  car-choose__category-items
  car-choose__category-name
  car-choose__head
  car_choose_container
  conditional-search__check-item
  conditional-search__check-list
  conditional-search__color-item
  conditional-search__color-list
  conditional-search__select-list
  container
  find-car_container
  find-cars
  find-cars__head
SNIPPET:
nect.facebook.net/en_US/fbevents.js"></script><script type="text/javascript" async="" src="https://www.googletagmanager.com/gtag/js?id=G-PTG3284JY2&amp;cx=c&amp;gtm=4e65d0"></script><script async="" src="https://scripts.clarity.ms/0.8.64/clarity.js"

In [16]:
script = """
import re
from playwright.sync_api import sync_playwright

with sync_playwright() as p:
    browser = p.chromium.launch(headless=True)
    page = browser.new_page(user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36")
    page.goto("https://www.sbtjapan.com/used-cars", timeout=60000, wait_until="domcontentloaded")
    page.wait_for_timeout(5000)
    html = page.content()
    print("TITLE:", page.title())
    print("URL:", page.url)
    print("HTML_LEN:", len(html))

    # Find relevant class names
    keywords = ["car", "vehicle", "stock", "list", "item", "card", "product", "result"]
    found = set()
    for kw in keywords:
        for m in re.findall(rf'class="([^"]*{kw}[^"]*)"', html, re.IGNORECASE):
            for cls in m.split():
                found.add(cls)
    print("\\nCLASSES:")
    for c in sorted(found)[:40]:
        print(" ", c)

    print("\\nSNIPPET:")
    print(html[3000:6000])
    browser.close()
"""

result = subprocess.run(
    [sys.executable, "-c", script],
    capture_output=True, text=True, timeout=120
)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr[-1000:])

TITLE: Search Used Cars For Sale | Best Quality Japanese Used Cars For Sale - SBT Japan
URL: https://www.sbtjapan.com/used-cars
HTML_LEN: 515432

CLASSES:
  -brands
  -filter
  -four-items
  -location
  -locations
  -parters-stock
  -sbt-stock
  -select
  -three-items
  -types
  breadcrumb__item
  breadcrumb__list
  car-choose
  car-choose__category
  car-choose__category-head
  car-choose__category-image
  car-choose__category-info
  car-choose__category-inner
  car-choose__category-item
  car-choose__category-items
  car-choose__category-name
  car-choose__head
  card-model
  card-model__company
  card-model__content
  card-model__image
  card-model__name-block
  card-model__product
  card-model__product-label
  card-model__total-search-hits
  conditional-search__check-item
  conditional-search__check-list
  conditional-search__color-item
  conditional-search__color-list
  conditional-search__select-list
  container
  find-cars
  find-cars__head
  find-cars__head-block
  find-cars__s

In [17]:
script = """
import re
from playwright.sync_api import sync_playwright

with sync_playwright() as p:
    browser = p.chromium.launch(headless=True)
    page = browser.new_page(user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36")
    page.goto("https://www.sbtjapan.com/used-cars", timeout=60000, wait_until="domcontentloaded")
    page.wait_for_timeout(5000)

    # Try card-model selector
    cards = page.query_selector_all(".card-model")
    print(f"card-model count: {len(cards)}")

    if cards:
        print("\\n--- First card HTML ---")
        print(cards[0].inner_html()[:2000])
    else:
        # Try to find what contains the actual listings
        html = page.content()
        # Find card-model in raw html
        idx = html.find("card-model")
        if idx > -1:
            print("Found card-model at index:", idx)
            print(html[idx-200:idx+1000])
        else:
            print("card-model not found in HTML")
            # Check what links exist to actual car detail pages
            links = page.eval_on_selector_all("a[href]", "els => els.map(e => e.href)")
            car_links = [l for l in links if "/used-cars/" in l and l.count("/") > 4]
            print(f"\\nCar detail links found: {len(car_links)}")
            for l in car_links[:10]:
                print(" ", l)

    browser.close()
"""

result = subprocess.run(
    [sys.executable, "-c", script],
    capture_output=True, text=True, timeout=120
)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr[-1000:])

card-model count: 10

--- First card HTML ---

                <div class="card-model__image">
                                                                                                        <img src="/html/template/default/assets/img/model/image___MAZDA___CX5.jpg" alt="CX5">
                </div>
                <div class="card-model__content">
                    <div class="card-model__name-block">
                        <div class="card-model__company">MAZDA</div>
                        <div class="card-model__product-label">
                            <p class="card-model__product">CX5</p>
                            <p class="card-model__total-search-hits">(1,811)</p>
                        </div>
                    </div>
                </div>
            



In [18]:
script = """
from playwright.sync_api import sync_playwright

with sync_playwright() as p:
    browser = p.chromium.launch(headless=True)
    page = browser.new_page(user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36")
    page.goto("https://www.sbtjapan.com/used-cars", timeout=60000, wait_until="domcontentloaded")
    page.wait_for_timeout(5000)

    # Get the href from the first card-model link
    cards = page.query_selector_all(".card-model")
    print(f"Total model cards: {len(cards)}")
    for card in cards[:5]:
        parent = card.evaluate("el => el.closest('a') ? el.closest('a').href : el.parentElement.closest('a') ? el.parentElement.closest('a').href : 'no link'")
        name = card.query_selector(".card-model__company")
        model = card.query_selector(".card-model__product")
        print(f"  {name.inner_text() if name else '?'} {model.inner_text() if model else '?'} -> {parent}")

    browser.close()
"""

result = subprocess.run(
    [sys.executable, "-c", script],
    capture_output=True, text=True, timeout=120
)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr[-1000:])

Total model cards: 10
  MAZDA CX5 -> https://www.sbtjapan.com/used-cars/search?make_id=5&model%5B%5D=CX5&isModel=1
  TOYOTA LAND CRUISER PRADO -> https://www.sbtjapan.com/used-cars/search?make_id=2&model%5B%5D=LAND%20CRUISER%20PRADO&isModel=1
  SUBARU FORESTER -> https://www.sbtjapan.com/used-cars/search?make_id=7&model%5B%5D=FORESTER&isModel=1
  HONDA VEZEL -> https://www.sbtjapan.com/used-cars/search?make_id=4&model%5B%5D=VEZEL&isModel=1
  NISSAN NOTE -> https://www.sbtjapan.com/used-cars/search?make_id=3&model%5B%5D=NOTE&isModel=1



In [19]:
script = """
import re
from playwright.sync_api import sync_playwright

with sync_playwright() as p:
    browser = p.chromium.launch(headless=True)
    page = browser.new_page(user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36")
    page.goto("https://www.sbtjapan.com/used-cars/search?make_id=5&model[]=CX5&isModel=1", timeout=60000, wait_until="domcontentloaded")
    page.wait_for_timeout(5000)

    print("TITLE:", page.title())
    print("URL:", page.url)

    html = page.content()
    print("HTML_LEN:", len(html))

    # Find all class names
    keywords = ["car", "vehicle", "stock", "list", "item", "card", "result", "search"]
    found = set()
    for kw in keywords:
        for m in re.findall(rf'class="([^"]*{kw}[^"]*)"', html, re.IGNORECASE):
            for cls in m.split():
                found.add(cls)
    print("\\nCLASSES:")
    for c in sorted(found)[:40]:
        print(" ", c)

    # Print a snippet where listings likely appear
    for marker in ["search-result", "car-list", "stock-list", "vehicle-list", "result-item"]:
        idx = html.find(marker)
        if idx > -1:
            print(f"\\nFound '{marker}' at {idx}:")
            print(html[idx:idx+1500])
            break

    browser.close()
"""

result = subprocess.run(
    [sys.executable, "-c", script],
    capture_output=True, text=True, timeout=120
)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr[-1000:])

TITLE: MAZDA CX5 Used Cars – Affordable, High-Quality Japanese Cars Worldwide | SBT Japan
URL: https://www.sbtjapan.com/used-cars/search?make_id=5&model[]=CX5&isModel=1
HTML_LEN: 3642301

CLASSES:
  -beige
  -black
  -blue
  -body-color
  -brown
  -calculate
  -detail
  -door
  -drive-type
  -engine-capacity
  -filter
  -four-items
  -fuel-type
  -gold
  -gray
  -green
  -horizontal
  -mileage
  -model-code
  -other
  -parters-stock
  -pearl
  -purple
  -red
  -row
  -sale
  -sbt-stock
  -search
  -search_result
  -seats
  -secondary
  -select
  -silver
  -steering-type
  -three-items
  -transmission
  -used_cars_search
  -white
  -yellow
  bottom-line-items

Found 'search-result' at 293755:
search-result">
							<div class="search-result__head">
								<div class="search-result__head-inner">
									<div class="search-result__head-block">
										<div class="search-result__result-number-block">
											<span class="search-result__result-number">1,813</span>
											<a href=

In [20]:
script = """
from playwright.sync_api import sync_playwright

with sync_playwright() as p:
    browser = p.chromium.launch(headless=True)
    page = browser.new_page(user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36")
    page.goto("https://www.sbtjapan.com/used-cars/search?make_id=5&model[]=CX5&isModel=1", timeout=60000, wait_until="domcontentloaded")
    page.wait_for_timeout(5000)

    html = page.content()

    # Find the area after search_result_area and print it
    idx = html.find("search_result_area")
    if idx > -1:
        print(html[idx:idx+3000])

    browser.close()
"""

result = subprocess.run(
    [sys.executable, "-c", script],
    capture_output=True, text=True, timeout=120
)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr[-1000:])

search_result_area">
                <div class="container">
					<div class="find-cars -search">
						<div class="search-result">
							<div class="search-result__head">
								<div class="search-result__head-inner">
									<div class="search-result__head-block">
										<div class="search-result__result-number-block">
											<span class="search-result__result-number">1,813</span>
											<a href="#search_result_area">Results</a>
                                        </div>
										<button class="search-result__filter-button" type="button" data-modal-trigger="filter">Filter</button>
                                                                                    <button class="button -secondary search-result__price-calculator" type="button" data-modal-trigger="price-calculator">Price Calculator</button>
                                        									</div>
                                    <div class="search-result__opiton-inner">
										<div class="search-

In [21]:
script = """
from playwright.sync_api import sync_playwright

with sync_playwright() as p:
    browser = p.chromium.launch(headless=True)
    page = browser.new_page(user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36")
    page.goto("https://www.sbtjapan.com/used-cars/search?make_id=5&model[]=CX5&isModel=1&show_numbers=20", timeout=60000, wait_until="domcontentloaded")
    page.wait_for_timeout(5000)

    html = page.content()

    # Find first actual car entry after the results head
    for marker in ["car-item", "car-card", "stock-item", "result__item", "search-result__item", "vehicle-item", "used-car"]:
        idx = html.find(marker)
        if idx > -1:
            print(f"Found '{marker}' at {idx}:")
            print(html[idx:idx+2000])
            print()
            break

    # Also try querying likely selectors
    for selector in [".car-item", ".car-card", ".search-result__item", ".result__item", ".stock-item", "ul.search-result__list li", ".used-cars-item"]:
        count = len(page.query_selector_all(selector))
        if count > 0:
            print(f"Selector '{selector}' → {count} elements")
            first = page.query_selector(selector)
            print(first.inner_html()[:1500])
            break

    browser.close()
"""

result = subprocess.run(
    [sys.executable, "-c", script],
    capture_output=True, text=True, timeout=120
)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr[-1000:])

Found 'result__item' at 296177:
result__items-block">
												<div class="select">
													<select class="select__field -is-selected" id="items" name="show_numbers">
														<option value="">---</option>
														 															<option value="20" selected="">20</option>
																													<option value="50">50</option>
																													<option value="100">100</option>
																											</select>
												</div>
											</div>
										</div>
									</div>
								</div>

                                <div class="pagination">
                                      
        <div class="pagination">
                            <a class="pagination__link -prev -is-disabled" href="javascript:void(0);"></a>
        
        <div class="pagination__inner">
                        <a class="pagination__link -is-current" href="/used-cars/search?make_id=5&amp;model%5B0%5D=CX5&amp;isModel=1&amp;show_numbers=20&amp;page=1">1</a>

      

In [22]:
script = """
from playwright.sync_api import sync_playwright

with sync_playwright() as p:
    browser = p.chromium.launch(headless=True)
    page = browser.new_page(user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36")
    page.goto("https://www.sbtjapan.com/used-cars/search?make_id=5&model[]=CX5&isModel=1&show_numbers=20", timeout=60000, wait_until="domcontentloaded")
    page.wait_for_timeout(5000)

    html = page.content()

    # Print what is inside search-result__inner
    idx = html.find("search-result__inner")
    if idx > -1:
        print(html[idx:idx+3000])

    browser.close()
"""

result = subprocess.run(
    [sys.executable, "-c", script],
    capture_output=True, text=True, timeout=120
)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr[-500:])

search-result__inner">
                                <form id="mainFilterForm" action="/used-cars/search" method="GET">
                                    <div class="search-result__aside">
                                                                                    <input type="hidden" name="isModel" value="1">
                                                                                                                                                                                                            <div class="conditional-search -filter" id="price-calculator" data-modal-content="price-calculator">
                                                <div class="conditional-search__head-inner"><button class="modal__close" type="button"></button></div>
                                                <div class="conditional-search__form -filter">
                                                <div class="conditional-search__head">Price Calculator</div>
                

In [23]:
script = """
from playwright.sync_api import sync_playwright

with sync_playwright() as p:
    browser = p.chromium.launch(headless=True)
    page = browser.new_page(user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36")
    page.goto("https://www.sbtjapan.com/used-cars/search?make_id=5&model[]=CX5&isModel=1&show_numbers=20", timeout=60000, wait_until="networkidle")
    page.wait_for_timeout(6000)

    html = page.content()

    # Look for any anchor links to individual car detail pages
    import re
    car_links = re.findall(r'href="(/used-cars/[^"]+/\d+[^"]*)"', html)
    print(f"Car detail links found: {len(car_links)}")
    for l in car_links[:5]:
        print(" ", l)

    # Print last 3000 chars of html where cards likely are
    print("\\n--- HTML tail ---")
    print(html[-3000:])

    browser.close()
"""

result = subprocess.run(
    [sys.executable, "-c", script],
    capture_output=True, text=True, timeout=120
)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr[-500:])

Car detail links found: 0

--- HTML tail ---
cbc3-43eb-a82b-c1eb4cecc649&amp;session_key=1deb6795-e6de-4e62-aaf5-5ab7b815f6cf&amp;storage_user_id=e53e3800-0d20-4770-b1c6-9ebe0789f663&amp;storage_session_id=91df5d73-226b-4a27-be74-c735c98bca58&amp;path=https%3A%2F%2Fwww.sbtjapan.com%2Fused-cars%2Fsearch%3Fmake_id%3D5%26model%5B%5D%3DCX5%26isModel%3D1%26show_numbers%3D20&amp;referrer=&amp;request_id=fbcb56f0-5c1a-4b27-8a94-79ed15a750d7&amp;at=2026-05-16T14:19:19.569Z&amp;language=en-US&amp;country_id=26&amp;items=%5B%7B%22sku%22%3A%22AI8066%22%7D%2C%7B%22sku%22%3A%22AI5495%22%7D%2C%7B%22sku%22%3A%22AI6543%22%7D%2C%7B%22sku%22%3A%22AI5745%22%7D%2C%7B%22sku%22%3A%22AH4754%22%7D%2C%7B%22sku%22%3A%22AH8954%22%7D%2C%7B%22sku%22%3A%22AI4551%22%7D%2C%7B%22sku%22%3A%22AI5097%22%7D%2C%7B%22sku%22%3A%22AI5173%22%7D%2C%7B%22sku%22%3A%22AI4776%22%7D%2C%7B%22sku%22%3A%22AH6139%22%7D%2C%7B%22sku%22%3A%22AH9767%22%7D%2C%7B%22sku%22%3A%22AH5888%22%7D%2C%7B%22sku%22%3A%22AI4079%22%7D%2C%7B%22sku%22%3A%22

In [25]:
script = """
from playwright.sync_api import sync_playwright

requests_log = []

with sync_playwright() as p:
    browser = p.chromium.launch(headless=True)
    page = browser.new_page(user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36")

    def on_request(request):
        url = request.url
        if any(kw in url.lower() for kw in ["api", "search", "stock", "cars", "json", "ajax", "vehicle"]):
            requests_log.append(f"{request.method} {url}")

    page.on("request", on_request)

    page.goto("https://www.sbtjapan.com/used-cars/search?make_id=5&model[]=CX5&isModel=1&show_numbers=20", timeout=60000, wait_until="domcontentloaded")
    page.wait_for_timeout(8000)

    print(f"Relevant requests: {len(requests_log)}")
    for r in requests_log:
        print(" ", r)

    browser.close()
"""

result = subprocess.run(
    [sys.executable, "-c", script],
    capture_output=True, text=True, timeout=120
)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr[-500:])

Relevant requests: 38
  GET https://www.sbtjapan.com/used-cars/search?make_id=5&model[]=CX5&isModel=1&show_numbers=20
  GET https://www.sbtjapan.com/used-cars/search?make_id=5&model[]=CX5&isModel=1&show_numbers=20
  GET https://fonts.googleapis.com/css2?family=Noto+Sans+JP:wght@100..900&family=Raleway:ital,wght@0,100..900;1,100..900&family=Roboto:ital,wght@0,100;0,300;0,400;0,500;0,700;0,900;1,100;1,300;1,400;1,500;1,700;1,900&family=Zen+Kaku+Gothic+Antique&family=Zen+Kaku+Gothic+New&display=swap
  GET https://www.sbtjapan.com/html/template/default/assets/css/find-cars.css?v=20260514E3cus5
  GET https://www.sbtjapan.com/html/template/default/assets/css/pickup-cars.css?v=20260514E3cus5
  GET https://api.onboarding-app.io/v1/onboarding-init?aid=218&pid=253&user_id=none&user_name=none&user_group_id=null&user_group_name=null
  GET https://www.sbtjapan.com/html/template/default/assets/img/icon_search.svg
  POST https://www.google.com/ccm/collect?rcb=2&frm=0&ae=g&en=page_view&dl=https%3A%2F%

In [26]:
script = """
from playwright.sync_api import sync_playwright

with sync_playwright() as p:
    browser = p.chromium.launch(headless=True)
    page = browser.new_page(user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36")
    page.goto("https://www.sbtjapan.com/used-cars/search?make_id=5&model[]=CX5&isModel=1&show_numbers=20", timeout=60000, wait_until="domcontentloaded")
    page.wait_for_timeout(5000)

    html = page.content()

    # Find where AI8066 (first SKU) appears in the HTML
    idx = html.find("AI8066")
    if idx > -1:
        print(f"Found SKU at index {idx}")
        print(html[idx-500:idx+1000])
    else:
        print("SKU not found in HTML")
        # Try scrolling and waiting longer
        page.evaluate("window.scrollTo(0, document.body.scrollHeight)")
        page.wait_for_timeout(3000)
        html2 = page.content()
        idx2 = html2.find("AI8066")
        print(f"After scroll - SKU found: {idx2 > -1}")
        if idx2 > -1:
            print(html2[idx2-500:idx2+1000])

    browser.close()
"""

result = subprocess.run(
    [sys.executable, "-c", script],
    capture_output=True, text=True, timeout=120
)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr[-500:])

Found SKU at index 345662
                                                                                                                   <li class="search-result__item">
                                                    <div class="card-product -horizontal card-grid-wrap ">
                                                        <div class="card-product__favorite-icon-area">
                                                            <input class="card-product__favorite-icon" type="checkbox" name="favorite" value="AI8066" data-car-id="2244109" onclick="changeFavorite(this);">
                                                        </div>
                                                        <a class="card-product__wrap" href="/used-cars/AI8066">
                                                            <div class="card-product__image">
                                                                <img src="https://img.sbtjapan.com/img/carphoto/AI8066/normal/f.jpg?v=20260516

In [27]:
script = """
from playwright.sync_api import sync_playwright

with sync_playwright() as p:
    browser = p.chromium.launch(headless=True)
    page = browser.new_page(user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36")
    page.goto("https://www.sbtjapan.com/used-cars/search?make_id=5&model[]=CX5&isModel=1&show_numbers=20", timeout=60000, wait_until="domcontentloaded")
    page.wait_for_timeout(5000)

    cards = page.query_selector_all("li.search-result__item")
    print(f"Cards found: {len(cards)}")

    if cards:
        print("\\n--- First card full HTML ---")
        print(cards[0].inner_html())

    browser.close()
"""

result = subprocess.run(
    [sys.executable, "-c", script],
    capture_output=True, text=True, timeout=120
)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr[-500:])

Cards found: 20

--- First card full HTML ---

                                                    <div class="card-product -horizontal card-grid-wrap ">
                                                        <div class="card-product__favorite-icon-area">
                                                            <input class="card-product__favorite-icon" type="checkbox" name="favorite" value="AI8066" data-car-id="2244109" onclick="changeFavorite(this);">
                                                        </div>
                                                        <a class="card-product__wrap" href="/used-cars/AI8066">
                                                            <div class="card-product__image">
                                                                <img src="https://img.sbtjapan.com/img/carphoto/AI8066/normal/f.jpg?v=20260516&amp;imwidth=300" alt="" onerror="this.onerror=null;this.src='/html/template/default/assets/img/default_product_comingsoon.webp

In [28]:
import re
import subprocess
import sys
import pandas as pd

MIN_YEAR = 2018
OUTPUT_CSV = "used_car_export_data.csv"

SEARCH_URLS = [
    "https://www.sbtjapan.com/used-cars/search?show_numbers=20&registration_year_from=2018",
]

script_template = r"""
import re, json
from playwright.sync_api import sync_playwright

MIN_YEAR = 2018
MAX_PAGES = {max_pages}
BASE_URL = "{base_url}"
results = []

def safe(card, selector):
    try:
        el = card.query_selector(selector)
        return el.inner_text().strip() if el else None
    except:
        return None

def parse_card(card):
    try:
        title = safe(card, ".card-product__product")
        year_match = re.search(r"\b(20\d{{2}})\b", title or "")
        year = int(year_match.group(1)) if year_match else None
        if year and year < MIN_YEAR:
            return None

        stock_id = safe(card, ".card-product__stock-value")
        link_el = card.query_selector("a.card-product__wrap")
        link = "https://www.sbtjapan.com" + link_el.get_attribute("href") if link_el else None

        return {{
            "stock_id":     stock_id.strip() if stock_id else None,
            "title":        title,
            "year":         year,
            "price_usd":    safe(card, ".card-product__vehicle-price .card-product__price"),
            "total_price":  safe(card, ".card-product__total-price .card-product__price"),
            "mileage":      safe(card, ".card-product__status.-mileage"),
            "engine_cc":    safe(card, ".card-product__status.-engine-capacity"),
            "transmission": safe(card, ".card-product__status.-transmission"),
            "fuel_type":    safe(card, ".card-product__status.-fuel-type"),
            "drive_type":   safe(card, ".card-product__status.-drive-type"),
            "steering":     safe(card, ".card-product__status.-steering-type"),
            "color":        safe(card, ".card-product__status.-body-color"),
            "location":     safe(card, ".card-product__location-value"),
            "url":          link,
        }}
    except Exception as e:
        return None

with sync_playwright() as p:
    browser = p.chromium.launch(headless=True)
    page = browser.new_page(user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36")
    page.goto(BASE_URL, timeout=60000, wait_until="domcontentloaded")
    page.wait_for_timeout(4000)

    for page_num in range(1, MAX_PAGES + 1):
        print(f"PAGE {{page_num}}", flush=True)
        try:
            page.wait_for_selector("li.search-result__item", timeout=10000)
        except:
            print("NO_CARDS", flush=True)
            break

        cards = page.query_selector_all("li.search-result__item")
        print(f"CARDS {{len(cards)}}", flush=True)

        for card in cards:
            car = parse_card(card)
            if car:
                results.append(car)

        # paginate
        next_btn = page.query_selector("a.pagination__link.-next:not(.-is-disabled)")
        if not next_btn:
            print("NO_NEXT", flush=True)
            break
        next_btn.click()
        page.wait_for_timeout(3000)

    browser.close()

print("RESULTS:" + json.dumps(results))
"""

def scrape(url, max_pages=5):
    script = script_template.format(base_url=url, max_pages=max_pages)
    result = subprocess.run(
        [sys.executable, "-c", script],
        capture_output=True, text=True, timeout=300
    )
    cars = []
    for line in result.stdout.splitlines():
        print(line)
        if line.startswith("RESULTS:"):
            import json
            cars = json.loads(line[8:])
    if result.stderr:
        print("STDERR:", result.stderr[-500:])
    return cars

print("Running scraper — this will take ~30 seconds per page...")
all_cars = scrape(SEARCH_URLS[0], max_pages=5)
print(f"\nTotal cars scraped: {len(all_cars)}")

Running scraper — this will take ~30 seconds per page...
PAGE 1
CARDS 20
PAGE 2
CARDS 20
PAGE 3
CARDS 20
PAGE 4
CARDS 20
PAGE 5
CARDS 20
RESULTS:[{"stock_id": "AJ0311", "title": "1998/3 MITSUBISHI ROSA 29SEATER", "year": null, "price_usd": "15,770", "total_price": "21,993", "mileage": "187,000km", "engine_cc": "4,890cc", "transmission": "6MT", "fuel_type": "DIESEL", "drive_type": "2WD", "steering": "RHD", "color": "GRAY(L)", "location": "YOKOHAMA, JAPAN", "url": "https://www.sbtjapan.com/used-cars/AJ0311"}, {"stock_id": "AJ0681", "title": "2018 HONDA VEZEL HYBRID X HONDA SENSING", "year": 2018, "price_usd": "8,100", "total_price": "9,904", "mileage": "141,000km", "engine_cc": "1,500cc", "transmission": "AT", "fuel_type": "HYBRID(PETROL)", "drive_type": "2WD", "steering": "RHD", "color": "BLACK", "location": "OSAKA, JAPAN", "url": "https://www.sbtjapan.com/used-cars/AJ0681"}, {"stock_id": "AI9197", "title": "2020/2 HONDA N VAN L HONDA SENSING", "year": 2020, "price_usd": "1,980", "total

In [29]:
import json

# Save to CSV
df = pd.DataFrame(all_cars)
df.drop_duplicates(subset="stock_id", inplace=True)
df.reset_index(drop=True, inplace=True)
df.to_csv(OUTPUT_CSV, index=False)

print(f"Saved {len(df)} cars to {OUTPUT_CSV}")
print(f"\nSample:")
print(df[["stock_id","title","year","price_usd","mileage","fuel_type","transmission"]].head(10).to_string())
print(f"\nColumns: {list(df.columns)}")
print(f"\nYear range: {df['year'].min()} - {df['year'].max()}")
print(f"Makes: {df['title'].str.extract(r'^\d+(?:/\d+)? (\w+)')[0].value_counts().head(10).to_string()}")

Saved 39 cars to used_car_export_data.csv

Sample:
  stock_id                                    title    year price_usd    mileage       fuel_type transmission
0   AJ0311          1998/3 MITSUBISHI ROSA 29SEATER     NaN    15,770  187,000km          DIESEL          6MT
1   AJ0681  2018 HONDA VEZEL HYBRID X HONDA SENSING  2018.0     8,100  141,000km  HYBRID(PETROL)           AT
2   AI9197       2020/2 HONDA N VAN L HONDA SENSING  2020.0     1,980  199,000km          PETROL           AT
3   AI6319             2022/5 TOYOTA TOWNACE VAN DX  2022.0     5,800  144,000km          PETROL          5MT
4   AI6077                    2021/5 TOYOTA YARIS X  2021.0     4,700  101,000km          PETROL           AT
5   AI6070                    2021/6 TOYOTA YARIS X  2021.0     5,000   91,000km          PETROL           AT
6   AI6131                    2021/5 TOYOTA YARIS X  2021.0     4,800  104,000km          PETROL           AT
7   AI6068                    2021/4 TOYOTA YARIS X  2021.0     5,100

In [30]:
print("Scraping 20 pages (~400 cars)... this will take ~2 minutes...")
all_cars = scrape(SEARCH_URLS[0], max_pages=20)

df = pd.DataFrame(all_cars)
df.drop_duplicates(subset="stock_id", inplace=True)
df.reset_index(drop=True, inplace=True)
df.to_csv(OUTPUT_CSV, index=False)

print(f"\nTotal unique cars saved: {len(df)}")
print(f"Year range: {df['year'].min()} - {df['year'].max()}")
print(f"\nTop makes:")
print(df['title'].str.extract(r'^\d+(?:/\d+)? (\w+)')[0].value_counts().head(10).to_string())
print(f"\nFuel types:")
print(df['fuel_type'].value_counts().to_string())
print(f"\nFile saved: {OUTPUT_CSV}")

Scraping 20 pages (~400 cars)... this will take ~2 minutes...
PAGE 1
CARDS 20
PAGE 2
CARDS 20
PAGE 3
CARDS 20
PAGE 4
CARDS 20
PAGE 5
CARDS 20
PAGE 6
CARDS 20
PAGE 7
CARDS 20
PAGE 8
CARDS 20
PAGE 9
CARDS 20
PAGE 10
CARDS 20
PAGE 11
CARDS 20
PAGE 12
CARDS 20
PAGE 13
CARDS 20
PAGE 14
CARDS 20
PAGE 15
CARDS 20
PAGE 16
CARDS 20
PAGE 17
CARDS 20
PAGE 18
CARDS 20
PAGE 19
CARDS 20
PAGE 20
CARDS 20
RESULTS:[{"stock_id": "AJ0311", "title": "1998/3 MITSUBISHI ROSA 29SEATER", "year": null, "price_usd": "15,770", "total_price": "21,993", "mileage": "187,000km", "engine_cc": "4,890cc", "transmission": "6MT", "fuel_type": "DIESEL", "drive_type": "2WD", "steering": "RHD", "color": "GRAY(L)", "location": "YOKOHAMA, JAPAN", "url": "https://www.sbtjapan.com/used-cars/AJ0311"}, {"stock_id": "AJ0681", "title": "2018 HONDA VEZEL HYBRID X HONDA SENSING", "year": 2018, "price_usd": "8,100", "total_price": "9,904", "mileage": "141,000km", "engine_cc": "1,500cc", "transmission": "AT", "fuel_type": "HYBRID(PETRO

In [31]:
script = """
from playwright.sync_api import sync_playwright

with sync_playwright() as p:
    browser = p.chromium.launch(headless=True)
    page = browser.new_page(user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36")
    page.goto("https://www.sbtjapan.com/used-cars/search", timeout=60000, wait_until="domcontentloaded")
    page.wait_for_timeout(4000)

    options = page.eval_on_selector_all(
        "select[name='make_id'] option",
        "els => els.map(e => e.value + '|' + e.innerText.trim())"
    )
    print(f"Total makes: {len(options)}")
    for o in options:
        print(o)

    browser.close()
"""

result = subprocess.run(
    [sys.executable, "-c", script],
    capture_output=True, text=True, timeout=120
)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr[-300:])

Total makes: 0



In [32]:
script = """
import re
from playwright.sync_api import sync_playwright

with sync_playwright() as p:
    browser = p.chromium.launch(headless=True)
    page = browser.new_page(user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36")
    page.goto("https://www.sbtjapan.com/used-cars/search", timeout=60000, wait_until="domcontentloaded")
    page.wait_for_timeout(5000)

    html = page.content()

    # Search for make_id patterns in raw HTML
    matches = re.findall(r'make_id["\s:=]+(\d+)[^>]*>?\s*([A-Z][A-Z\s\-]+)', html)
    print("make_id matches:")
    for m in matches[:50]:
        print(f"  {m[0]} | {m[1].strip()}")

    # Also try finding option tags with values near make-related selects
    idx = html.find("make_id")
    if idx > -1:
        print("\\nHTML around make_id:")
        print(html[idx:idx+2000])

    browser.close()
"""

result = subprocess.run(
    [sys.executable, "-c", script],
    capture_output=True, text=True, timeout=120
)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr[-300:])

make_id matches:
  5 | CX
  2 | DO
  7 | ER
  4 | EL
  3 | TE

HTML around make_id:
make_id=">
<script>
    let urd = document.cookie.split('; ').find(row => row.startsWith('ud' + '='))?.split('=')[1] || 'none';
    let na = document.cookie.split('; ').find(row => row.startsWith('na' + '='))?.split('=')[1] || 'none';
    var userAgent = window.navigator.userAgent.toLowerCase();
    if ((userAgent.indexOf('msie') === -1 && userAgent.indexOf('trident') === -1) && !document.querySelector("#stands_onbd_point")) {
    var ONB = ONB || {};
    ONB.ignition_url = "https://api.onboarding-app.io/v1/onboarding-init?aid=218&pid=253";
    // Custom Area Start=====================
    ONB._queryparam = {
        "user_id": urd,
        "user_name": na,
        "user_group_id": null,
        "user_group_name": null
    }
    ONB.black_list = [];
    ONB._custom_functions = {};
    // Custom Area End======================

    ONB.embed = function(){for(ONB.item in ONB._queryparam){ONB.ignition_url+=

In [33]:
script = """
from playwright.sync_api import sync_playwright

# Test known European make URLs directly
test_makes = [
    ("MERCEDES", "https://www.sbtjapan.com/used-cars/search?make_id=20&show_numbers=20"),
    ("BMW",      "https://www.sbtjapan.com/used-cars/search?make_id=21&show_numbers=20"),
    ("MERCEDES", "https://www.sbtjapan.com/used-cars/search?make_id=22&show_numbers=20"),
    ("BMW",      "https://www.sbtjapan.com/used-cars/search?make_id=23&show_numbers=20"),
    ("AUDI",     "https://www.sbtjapan.com/used-cars/search?make_id=24&show_numbers=20"),
    ("VW",       "https://www.sbtjapan.com/used-cars/search?make_id=25&show_numbers=20"),
    ("VOLVO",    "https://www.sbtjapan.com/used-cars/search?make_id=26&show_numbers=20"),
    ("LEXUS",    "https://www.sbtjapan.com/used-cars/search?make_id=27&show_numbers=20"),
]

with sync_playwright() as p:
    browser = p.chromium.launch(headless=True)
    page = browser.new_page(user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36")

    for label, url in test_makes:
        page.goto(url, timeout=60000, wait_until="domcontentloaded")
        page.wait_for_timeout(3000)
        title = page.title()
        count_el = page.query_selector(".search-result__result-number")
        count = count_el.inner_text().strip() if count_el else "?"
        print(f"make_id={url.split('make_id=')[1].split('&')[0]:3s} | {count:>8s} results | {title[:60]}")

    browser.close()
"""

result = subprocess.run(
    [sys.executable, "-c", script],
    capture_output=True, text=True, timeout=120
)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr[-300:])

make_id=20  |       95 results | DODGE Used Cars – Affordable, High-Quality Japanese Cars Wor
make_id=21  |    3,467 results | FORD Used Cars – Affordable, High-Quality Japanese Cars Worl
make_id=22  |        2 results | HUMMER Used Cars – Affordable, High-Quality Japanese Cars Wo
make_id=23  |      280 results | LINCOLN Used Cars – Affordable, High-Quality Japanese Cars W
make_id=24  |        0 results | MERCURY Used Cars – Affordable, High-Quality Japanese Cars W
make_id=25  |        0 results | R)PONTIAC Used Cars – Affordable, High-Quality Japanese Cars
make_id=26  |        0 results | SATURN Used Cars – Affordable, High-Quality Japanese Cars Wo
make_id=27  |        0 results | STAR CRAFT Used Cars – Affordable, High-Quality Japanese Car



In [34]:
script = """
from playwright.sync_api import sync_playwright

with sync_playwright() as p:
    browser = p.chromium.launch(headless=True)
    page = browser.new_page(user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36")
    page.goto("https://www.sbtjapan.com/used-cars/search?show_numbers=100&registration_year_from=2018", timeout=60000, wait_until="domcontentloaded")
    page.wait_for_timeout(4000)

    total = page.query_selector(".search-result__result-number")
    last_page = page.query_selector_all(".pagination__link")
    
    print("Total results:", total.inner_text() if total else "?")
    
    # Get last page number
    page_nums = []
    for el in last_page:
        t = el.inner_text().strip()
        if t.isdigit():
            page_nums.append(int(t))
    print("Last page number:", max(page_nums) if page_nums else "?")
    print("Cars per page: 100")
    print("Est total cars with year>=2018:", total.inner_text() if total else "?")

    browser.close()
"""

result = subprocess.run(
    [sys.executable, "-c", script],
    capture_output=True, text=True, timeout=120
)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr[-300:])

Total results: 286,997
Last page number: 2870
Cars per page: 100
Est total cars with year>=2018: 286,997



In [35]:
print("Scraping 500 cars (5 pages x 100 cars)...")
all_cars = scrape(
    "https://www.sbtjapan.com/used-cars/search?show_numbers=100&registration_year_from=2018",
    max_pages=5
)

df = pd.DataFrame(all_cars)
df.drop_duplicates(subset="stock_id", inplace=True)
df.reset_index(drop=True, inplace=True)
df.to_csv(OUTPUT_CSV, index=False)

print(f"\nTotal unique cars saved: {len(df)}")
print(f"File: {OUTPUT_CSV}")
print(df[["stock_id","title","year","price_usd","mileage","fuel_type"]].head(10).to_string())

Scraping 500 cars (5 pages x 100 cars)...


TimeoutExpired: Command '['C:\\Users\\USER\\anaconda3\\python.exe', '-c', '\nimport re, json\nfrom playwright.sync_api import sync_playwright\n\nMIN_YEAR = 2018\nMAX_PAGES = 5\nBASE_URL = "https://www.sbtjapan.com/used-cars/search?show_numbers=100&registration_year_from=2018"\nresults = []\n\ndef safe(card, selector):\n    try:\n        el = card.query_selector(selector)\n        return el.inner_text().strip() if el else None\n    except:\n        return None\n\ndef parse_card(card):\n    try:\n        title = safe(card, ".card-product__product")\n        year_match = re.search(r"\\b(20\\d{2})\\b", title or "")\n        year = int(year_match.group(1)) if year_match else None\n        if year and year < MIN_YEAR:\n            return None\n\n        stock_id = safe(card, ".card-product__stock-value")\n        link_el = card.query_selector("a.card-product__wrap")\n        link = "https://www.sbtjapan.com" + link_el.get_attribute("href") if link_el else None\n\n        return {\n          

In [36]:
def scrape(url, max_pages=5, timeout=600):
    script = script_template.format(base_url=url, max_pages=max_pages)
    result = subprocess.run(
        [sys.executable, "-c", script],
        capture_output=True, text=True, timeout=timeout
    )
    cars = []
    for line in result.stdout.splitlines():
        print(line)
        if line.startswith("RESULTS:"):
            import json
            cars = json.loads(line[8:])
    if result.stderr:
        print("STDERR:", result.stderr[-500:])
    return cars

print("Scraping 500 cars (25 pages x 20 cars)...")
all_cars = scrape(
    "https://www.sbtjapan.com/used-cars/search?show_numbers=20&registration_year_from=2018",
    max_pages=25,
    timeout=600
)

df = pd.DataFrame(all_cars)
df.drop_duplicates(subset="stock_id", inplace=True)
df.reset_index(drop=True, inplace=True)
df.to_csv(OUTPUT_CSV, index=False)

print(f"\nTotal unique cars saved: {len(df)}")
print(f"File: {OUTPUT_CSV}")
print(df[["stock_id","title","year","price_usd","mileage","fuel_type"]].head(10).to_string())

Scraping 500 cars (25 pages x 20 cars)...
PAGE 1
CARDS 20
PAGE 2
CARDS 20
PAGE 3
CARDS 20
PAGE 4
CARDS 20
PAGE 5
CARDS 20
PAGE 6
CARDS 20
PAGE 7
CARDS 20
PAGE 8
CARDS 20
PAGE 9
CARDS 20
PAGE 10
CARDS 20
PAGE 11
CARDS 20
PAGE 12
CARDS 20
PAGE 13
CARDS 20
PAGE 14
CARDS 20
PAGE 15
CARDS 20
PAGE 16
CARDS 20
PAGE 17
CARDS 20
PAGE 18
CARDS 20
PAGE 19
CARDS 20
PAGE 20
CARDS 20
PAGE 21
CARDS 20
PAGE 22
CARDS 20
PAGE 23
CARDS 20
PAGE 24
CARDS 20
PAGE 25
CARDS 20
RESULTS:[{"stock_id": "AJ0311", "title": "1998/3 MITSUBISHI ROSA 29SEATER", "year": null, "price_usd": "15,770", "total_price": "21,993", "mileage": "187,000km", "engine_cc": "4,890cc", "transmission": "6MT", "fuel_type": "DIESEL", "drive_type": "2WD", "steering": "RHD", "color": "GRAY(L)", "location": "YOKOHAMA, JAPAN", "url": "https://www.sbtjapan.com/used-cars/AJ0311"}, {"stock_id": "AJ0681", "title": "2018 HONDA VEZEL HYBRID X HONDA SENSING", "year": 2018, "price_usd": "8,100", "total_price": "9,904", "mileage": "141,000km", "engine

In [37]:
import os

files = ["used_car_export_data.csv", "used_car_export_data_full.csv"]
for f in files:
    if os.path.exists(f):
        size = os.path.getsize(f)
        print(f"Found: {f}")
        print(f"  Path: {os.path.abspath(f)}")
        print(f"  Size: {size/1024:.1f} KB")

Found: used_car_export_data.csv
  Path: C:\Users\USER\anaconda_projects\b1547a7e-c835-4a58-9c35-af0057c878a0\used_car_export_data.csv
  Size: 23.2 KB


In [38]:
df = pd.read_csv("used_car_export_data.csv")
print(df.columns.tolist())
print(df[["stock_id","title","price_usd","total_price"]].head(10).to_string())

['stock_id', 'title', 'year', 'price_usd', 'total_price', 'mileage', 'engine_cc', 'transmission', 'fuel_type', 'drive_type', 'steering', 'color', 'location', 'url']
  stock_id                                    title price_usd total_price
0   AJ0311          1998/3 MITSUBISHI ROSA 29SEATER    15,770      21,993
1   AJ0681  2018 HONDA VEZEL HYBRID X HONDA SENSING     8,100       9,904
2   AI9197       2020/2 HONDA N VAN L HONDA SENSING     1,980       3,400
3   AI6319             2022/5 TOYOTA TOWNACE VAN DX     5,800       7,579
4   AI6077                    2021/5 TOYOTA YARIS X     4,700       6,149
5   AI6070                    2021/6 TOYOTA YARIS X     5,000       6,453
6   AI6131                    2021/5 TOYOTA YARIS X     4,800       6,250
7   AI6068                    2021/4 TOYOTA YARIS X     5,100       6,554
8   AI6132                    2021/4 TOYOTA YARIS X     4,800       6,250
9   AI6134                    2021/6 TOYOTA YARIS X     4,800       6,250


In [41]:
df.describe

<bound method NDFrame.describe of     stock_id                                              title    year  \
0     AJ0311                    1998/3 MITSUBISHI ROSA 29SEATER     NaN   
1     AJ0681            2018 HONDA VEZEL HYBRID X HONDA SENSING  2018.0   
2     AI9197                 2020/2 HONDA N VAN L HONDA SENSING  2020.0   
3     AI6319                       2022/5 TOYOTA TOWNACE VAN DX  2022.0   
4     AI6077                              2021/5 TOYOTA YARIS X  2021.0   
..       ...                                                ...     ...   
137   AI5625                2018/3 DAIHATSU HIJET CARGO SPECIAL  2018.0   
138   AI5599                2019/3 TOYOTA COROLLA AXIO HYBRID G  2019.0   
139   AH5056  2019/12 TOYOTA PROBOX VAN HYBRID HYBRID DX COM...  2019.0   
140   AH6431                         2019/2 TOYOTA PROBOX VAN F  2019.0   
141   AI5591                             2019/4 NISSAN SERENA B  2019.0   

    price_usd total_price    mileage engine_cc transmission      

In [42]:
import os, time, json

BASE_URL = "https://www.sbtjapan.com/used-cars/search?show_numbers=20&registration_year_from=2018&page={page}"
BATCH_SIZE = 50       # save every 50 pages
TOTAL_PAGES = 2870    # all pages on site
OUTPUT_CSV  = "used_car_export_data_full.csv"
PROGRESS_FILE = "scraper_progress.json"

# Load progress if resuming
if os.path.exists(PROGRESS_FILE):
    with open(PROGRESS_FILE) as f:
        progress = json.load(f)
    start_page = progress["last_page"] + 1
    all_cars   = progress["cars"]
    print(f"Resuming from page {start_page}, {len(all_cars)} cars already collected")
else:
    start_page = 1
    all_cars   = []
    print("Starting fresh scrape...")

batch_script = r"""
import re, json
from playwright.sync_api import sync_playwright

results = []

def safe(card, selector):
    try:
        el = card.query_selector(selector)
        return el.inner_text().strip() if el else None
    except:
        return None

def parse_card(card):
    try:
        title = safe(card, ".card-product__product")
        year_match = re.search(r"\b(20\d{2})\b", title or "")
        year = int(year_match.group(1)) if year_match else None
        stock_id = safe(card, ".card-product__stock-value")
        link_el = card.query_selector("a.card-product__wrap")
        link = "https://www.sbtjapan.com" + link_el.get_attribute("href") if link_el else None
        return {
            "stock_id":     stock_id.strip() if stock_id else None,
            "title":        title,
            "year":         year,
            "price_usd":    safe(card, ".card-product__vehicle-price .card-product__price"),
            "total_price":  safe(card, ".card-product__total-price .card-product__price"),
            "mileage":      safe(card, ".card-product__status.-mileage"),
            "engine_cc":    safe(card, ".card-product__status.-engine-capacity"),
            "transmission": safe(card, ".card-product__status.-transmission"),
            "fuel_type":    safe(card, ".card-product__status.-fuel-type"),
            "drive_type":   safe(card, ".card-product__status.-drive-type"),
            "steering":     safe(card, ".card-product__status.-steering-type"),
            "color":        safe(card, ".card-product__status.-body-color"),
            "location":     safe(card, ".card-product__location-value"),
            "url":          link,
        }
    except:
        return None

PAGES = {pages}

with sync_playwright() as p:
    browser = p.chromium.launch(headless=True)
    page = browser.new_page(user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36")

    for page_num in PAGES:
        url = "{base_url}".format(page=page_num)
        try:
            page.goto(url, timeout=60000, wait_until="domcontentloaded")
            page.wait_for_timeout(2500)
            page.wait_for_selector("li.search-result__item", timeout=10000)
            cards = page.query_selector_all("li.search-result__item")
            for card in cards:
                car = parse_card(card)
                if car:
                    results.append(car)
            print(f"PAGE {page_num} OK {len(cards)}", flush=True)
        except Exception as e:
            print(f"PAGE {page_num} ERR {e}", flush=True)

    browser.close()

print("RESULTS:" + json.dumps(results))
"""

def scrape_pages(page_list):
    script = batch_script.replace("{pages}", str(page_list)).replace("{base_url}", BASE_URL)
    result = subprocess.run(
        [sys.executable, "-c", script],
        capture_output=True, text=True, timeout=600
    )
    cars = []
    for line in result.stdout.splitlines():
        if line.startswith("PAGE"):
            print(line)
        elif line.startswith("RESULTS:"):
            cars = json.loads(line[8:])
    return cars

# Run in batches
batch_num = 0
for batch_start in range(start_page, TOTAL_PAGES + 1, BATCH_SIZE):
    batch_end = min(batch_start + BATCH_SIZE - 1, TOTAL_PAGES)
    pages = list(range(batch_start, batch_end + 1))
    batch_num += 1

    print(f"\nBatch {batch_num}: pages {batch_start}-{batch_end}")
    t0 = time.time()
    batch_cars = scrape_pages(pages)
    elapsed = time.time() - t0

    all_cars.extend(batch_cars)

    # Deduplicate and save
    df = pd.DataFrame(all_cars)
    df.drop_duplicates(subset="stock_id", inplace=True)
    df.to_csv(OUTPUT_CSV, index=False)

    # Save progress
    with open(PROGRESS_FILE, "w") as f:
        json.dump({"last_page": batch_end, "cars": df.to_dict("records")}, f)

    print(f"  Got {len(batch_cars)} cars | Total: {len(df)} | Time: {elapsed:.0f}s | CSV saved")

print(f"\nDONE! Total unique cars: {len(df)}")
print(f"File: {OUTPUT_CSV}")

Starting fresh scrape...

Batch 1: pages 1-50


TimeoutExpired: Command '['C:\\Users\\USER\\anaconda3\\python.exe', '-c', '\nimport re, json\nfrom playwright.sync_api import sync_playwright\n\nresults = []\n\ndef safe(card, selector):\n    try:\n        el = card.query_selector(selector)\n        return el.inner_text().strip() if el else None\n    except:\n        return None\n\ndef parse_card(card):\n    try:\n        title = safe(card, ".card-product__product")\n        year_match = re.search(r"\\b(20\\d{2})\\b", title or "")\n        year = int(year_match.group(1)) if year_match else None\n        stock_id = safe(card, ".card-product__stock-value")\n        link_el = card.query_selector("a.card-product__wrap")\n        link = "https://www.sbtjapan.com" + link_el.get_attribute("href") if link_el else None\n        return {\n            "stock_id":     stock_id.strip() if stock_id else None,\n            "title":        title,\n            "year":         year,\n            "price_usd":    safe(card, ".card-product__vehicle-price .

In [43]:
import os, time, json

BASE_URL = "https://www.sbtjapan.com/used-cars/search?show_numbers=20&registration_year_from=2018&page={page}"
BATCH_SIZE    = 25     # reduced from 50
TOTAL_PAGES   = 2870
OUTPUT_CSV    = "used_car_export_data_full.csv"
PROGRESS_FILE = "scraper_progress.json"

# Load progress if resuming
if os.path.exists(PROGRESS_FILE):
    with open(PROGRESS_FILE) as f:
        progress = json.load(f)
    start_page = progress["last_page"] + 1
    all_cars   = progress["cars"]
    print(f"Resuming from page {start_page}, {len(all_cars)} cars already collected")
else:
    start_page = 1
    all_cars   = []
    print("Starting fresh scrape...")

batch_script = r"""
import re, json
from playwright.sync_api import sync_playwright

results = []

def safe(card, selector):
    try:
        el = card.query_selector(selector)
        return el.inner_text().strip() if el else None
    except:
        return None

def parse_card(card):
    try:
        title = safe(card, ".card-product__product")
        year_match = re.search(r"\b(20\d{2})\b", title or "")
        year = int(year_match.group(1)) if year_match else None
        stock_id = safe(card, ".card-product__stock-value")
        link_el = card.query_selector("a.card-product__wrap")
        link = "https://www.sbtjapan.com" + link_el.get_attribute("href") if link_el else None
        return {
            "stock_id":     stock_id.strip() if stock_id else None,
            "title":        title,
            "year":         year,
            "price_usd":    safe(card, ".card-product__vehicle-price .card-product__price"),
            "total_price":  safe(card, ".card-product__total-price .card-product__price"),
            "mileage":      safe(card, ".card-product__status.-mileage"),
            "engine_cc":    safe(card, ".card-product__status.-engine-capacity"),
            "transmission": safe(card, ".card-product__status.-transmission"),
            "fuel_type":    safe(card, ".card-product__status.-fuel-type"),
            "drive_type":   safe(card, ".card-product__status.-drive-type"),
            "steering":     safe(card, ".card-product__status.-steering-type"),
            "color":        safe(card, ".card-product__status.-body-color"),
            "location":     safe(card, ".card-product__location-value"),
            "url":          link,
        }
    except:
        return None

PAGES = PAGES_PLACEHOLDER

with sync_playwright() as p:
    browser = p.chromium.launch(headless=True)
    page = browser.new_page(user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36")

    for page_num in PAGES:
        url = "BASE_URL_PLACEHOLDER".replace("{page}", str(page_num))
        try:
            page.goto(url, timeout=60000, wait_until="domcontentloaded")
            page.wait_for_timeout(2500)
            page.wait_for_selector("li.search-result__item", timeout=10000)
            cards = page.query_selector_all("li.search-result__item")
            for card in cards:
                car = parse_card(card)
                if car:
                    results.append(car)
            print(f"PAGE {page_num} OK {len(cards)}", flush=True)
        except Exception as e:
            print(f"PAGE {page_num} ERR {e}", flush=True)
            continue

    browser.close()

print("RESULTS:" + json.dumps(results))
"""

def scrape_pages(page_list):
    script = batch_script\
        .replace("PAGES_PLACEHOLDER", str(page_list))\
        .replace("BASE_URL_PLACEHOLDER", BASE_URL)
    result = subprocess.run(
        [sys.executable, "-c", script],
        capture_output=True, text=True, timeout=700  # increased timeout
    )
    cars = []
    for line in result.stdout.splitlines():
        if line.startswith("PAGE"):
            print(line)
        elif line.startswith("RESULTS:"):
            cars = json.loads(line[8:])
    if result.returncode != 0 and result.stderr:
        print(f"  STDERR: {result.stderr[-200:]}")
    return cars

# Run in batches
batch_num = 0
for batch_start in range(start_page, TOTAL_PAGES + 1, BATCH_SIZE):
    batch_end  = min(batch_start + BATCH_SIZE - 1, TOTAL_PAGES)
    pages      = list(range(batch_start, batch_end + 1))
    batch_num += 1

    print(f"\nBatch {batch_num}: pages {batch_start}-{batch_end}")
    t0 = time.time()

    try:
        batch_cars = scrape_pages(pages)
    except Exception as e:
        print(f"  Batch failed: {e} — skipping and continuing")
        batch_cars = []

    all_cars.extend(batch_cars)

    # Deduplicate and save
    df = pd.DataFrame(all_cars)
    if not df.empty:
        df.drop_duplicates(subset="stock_id", inplace=True)
        df.to_csv(OUTPUT_CSV, index=False)

        # Save progress
        with open(PROGRESS_FILE, "w") as f:
            json.dump({"last_page": batch_end, "cars": df.to_dict("records")}, f)

    elapsed = time.time() - t0
    print(f"  Got {len(batch_cars)} cars | Total: {len(df)} | Time: {elapsed:.0f}s | Saved")

print(f"\nDONE! Total unique cars: {len(df)}")
print(f"File: {OUTPUT_CSV}")

Starting fresh scrape...

Batch 1: pages 1-25
PAGE 1 OK 20
PAGE 2 OK 20
PAGE 3 OK 20
PAGE 4 OK 20
PAGE 5 OK 20
PAGE 6 OK 20
PAGE 7 OK 20
PAGE 8 OK 20
PAGE 9 OK 20
PAGE 10 OK 20
PAGE 11 OK 20
PAGE 12 OK 20
PAGE 13 OK 20
PAGE 14 OK 20
PAGE 15 OK 20
PAGE 16 OK 20
PAGE 17 OK 20
PAGE 18 OK 20
PAGE 19 OK 20
PAGE 20 OK 20
PAGE 21 OK 20
PAGE 22 OK 20
PAGE 23 OK 20
PAGE 24 OK 20
PAGE 25 OK 20
  Got 500 cars | Total: 500 | Time: 298s | Saved

Batch 2: pages 26-50
PAGE 26 OK 20
PAGE 27 OK 20
PAGE 28 OK 20
PAGE 29 OK 20
PAGE 30 OK 20
PAGE 31 OK 20
PAGE 32 OK 20
PAGE 33 OK 20
PAGE 34 OK 20
PAGE 35 OK 20
PAGE 36 OK 20
PAGE 37 OK 20
PAGE 38 OK 20
PAGE 39 OK 20
PAGE 40 OK 20
PAGE 41 OK 20
PAGE 42 OK 20
PAGE 43 OK 20
PAGE 44 OK 20
PAGE 45 OK 20
PAGE 46 OK 20
PAGE 47 OK 20
PAGE 48 OK 20
PAGE 49 OK 20
PAGE 50 OK 20
  Got 500 cars | Total: 1000 | Time: 481s | Saved

Batch 3: pages 51-75
PAGE 51 OK 20
PAGE 52 OK 20
PAGE 53 OK 20
PAGE 54 OK 20
PAGE 55 OK 20
PAGE 56 OK 20
PAGE 57 OK 20
PAGE 58 OK 20
PAGE 59 

In [44]:
import os

print("Working directory:")
print(os.getcwd())

print("\nFiles found:")
for f in ["used_car_export_data.csv", "used_car_export_data_full.csv", "scraper_progress.json", "README.md"]:
    if os.path.exists(f):
        size = os.path.getsize(f) / 1024
        print(f"  {f} — {size:.1f} KB — {os.path.abspath(f)}")
    else:
        print(f"  {f} — not found yet")

Working directory:
C:\Users\USER\anaconda_projects\b1547a7e-c835-4a58-9c35-af0057c878a0

Files found:
  used_car_export_data.csv — 15.0 KB — C:\Users\USER\anaconda_projects\b1547a7e-c835-4a58-9c35-af0057c878a0\used_car_export_data.csv
  used_car_export_data_full.csv — 6763.9 KB — C:\Users\USER\anaconda_projects\b1547a7e-c835-4a58-9c35-af0057c878a0\used_car_export_data_full.csv
  scraper_progress.json — 14899.6 KB — C:\Users\USER\anaconda_projects\b1547a7e-c835-4a58-9c35-af0057c878a0\scraper_progress.json
  README.md — not found yet


In [45]:
print("The end")

The end
